In [1]:
import pandas as pd

In [2]:
import h5py

In [3]:
h5_path = "/project2/fudenber_735/smaruj/akitaX1_analyses_data/virtual_insertion_singletons/model_0.h5"

In [4]:
with h5py.File(h5_path, "r") as f:
    print("Keys in HDF5 file:")
    for key in f.keys():
        print(key)

Keys in HDF5 file:
SCD_h1_m0
alt_INS-16_h1_m0
alt_INS-64_h1_m0
background_index
boundary_span
boundary_strength_200000
chrom
end
flank_bp
jaspar_score
log2_insulation_score_200000
orientation
ref_INS-16_h1_m0
ref_INS-64_h1_m0
seq_id
spacer_bp
start
strand


In [5]:
with h5py.File(h5_path, "r") as f:
    # SCD array: shape (n_sequences, n_targets)
    scd = f["SCD_h1_m0"][:]              
    background_index = f["background_index"][:]  # same length as n_sequences
    seq_id = f["seq_id"][:]                        # same length
    chrom = f["chrom"][:]
    start = f["start"][:]
    end = f["end"][:]
    strand = f["strand"][:]

In [6]:
# Decode bytes if needed
def decode_if_bytes(x):
    if isinstance(x[0], bytes):
        return [s.decode("utf-8") for s in x]
    return x

In [7]:
strand = decode_if_bytes(strand)
chrom = decode_if_bytes(chrom)

In [8]:
# Take first target
scd_target0 = scd[:, 0]

In [9]:
df = pd.DataFrame({
    "seq_id": seq_id,
    "background_index": background_index,
    "SCD_tg0": scd_target0,
    "chrom": chrom,
    "start": start,
    "end": end,
    "strand": strand
})

In [10]:
# Average SCD per sequence across backgrounds
df_avg = df.groupby(["seq_id", "chrom", "start", "end", "strand"])["SCD_tg0"].mean().reset_index()

In [13]:
# Sort descending by SCD (target 0)
df_top = df_avg.sort_values(by="SCD_tg0", ascending=False)

# Select top 100
df_top100 = df_top.head(100).reset_index(drop=True)

In [15]:
df_top100.to_csv("/scratch1/smaruj/full_akita_vs_semifreddo/top100_ctcfs.csv", index=False)